# 10a — RQ1 đánh giá cuối: train lại mô hình neural (Kaggle)

**Vị trí trong Giai đoạn 10**

Giai đoạn 10 gồm ba notebook: 10a (Kaggle) train lại các mô hình neural và dự đoán; 10b (máy local) gọi LLM; 10 (máy local)
chấm điểm Hold-out, focused set và Lazada đúng một lần. Notebook này không có nhãn của các tập khóa.

**Cấu hình đóng băng**

Mọi cấu hình, ngưỡng và seed được đọc từ `03_configs/frozen/` (amendment v2.1); không có bước tinh chỉnh nào. Tầng 1, tầng 2 và
baseline phẳng được train lại trong cùng một phiên với ba seed 42, 1337, 2024.

**Dữ liệu trên Kaggle**

Train và Validation có nhãn để train và kiểm tra. Hold-out, focused set và Lazada chỉ có `review_id` và `text`; notebook dừng
nếu phát hiện file có nhãn của các tập khóa trong `/kaggle/input`.

**Cổng Bước 0**

Trước khi ghi dự đoán của các tập khóa, Macro-F1 trên Validation của mô hình train lại được so với kết quả đã báo cáo ở GĐ6, GĐ7
và GĐ9. Mỗi mô hình phải lệch không quá max(2 điểm, 1 độ lệch chuẩn giữa seed của kết quả cũ), và recall ACTIONABLE của seed 42
ở ngưỡng đóng băng phải đạt sàn 0,80. Không đạt thì notebook dừng và không ghi dự đoán của các tập khóa.

In [ ]:
import os, subprocess, sys, json, gc, datetime
from kaggle_secrets import UserSecretsClient
REPO_URL = 'github.com/dnhphuong-05/kltn-vn-app-review-requirements.git'
REPO_DIR = '/kaggle/working/repo'
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{REPO_URL}', REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone thất bại: ' + r.stderr.replace(token, '***'))
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)
del token
sys.path.append(os.path.join(REPO_DIR, '02_src'))
import numpy as np, pandas as pd, torch, transformers, sentencepiece
from src.utils.seeds import set_seed
set_seed(42)
COMMIT = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('Ngày chạy:', datetime.date.today().isoformat(), '| commit:', COMMIT)
print('transformers', transformers.__version__, '| sentencepiece', sentencepiece.__version__, '| torch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có')

## Cấu hình đóng băng

**Nguồn cấu hình**

`load_frozen` đọc ba file `tier1_config.yaml`, `tier2_config.yaml`, `flat_config.yaml`, kiểm tra header FROZEN v2.1 và đối chiếu
prompt_hash với `04_prompts/prompts_registry.md`; mọi sai khác đều làm notebook dừng.

In [ ]:
from src.classification.final_eval import load_frozen, LOCKED, TEXT_ONLY
FZ = load_frozen(REPO_DIR)
SEEDS = FZ['tier1']['seeds']
assert SEEDS == FZ['tier2']['seeds'] == FZ['flat']['seeds'] == [42, 1337, 2024]
for tier, model in (('tier1', 'visobert'), ('tier1', 'bilstm'), ('tier2', 'visobert'), ('tier2', 'bilstm'), ('flat', 'visobert')):
    c = FZ[tier][model]['config']
    keys = {k: c[k] for k in ('lr', 'epochs', 'hidden_size', 'dropout') if k in c}
    thr = FZ[tier][model].get('threshold', '—')
    print(f'{tier:5s} {model:8s} {keys} | ngưỡng {thr}')
print('Sàn recall tầng 1:', FZ['tier1']['threshold_rule']['min_recall'])

## Dữ liệu

**Tập có nhãn**

Train 1.200 và Validation 350 đánh giá, đọc từ dataset Kaggle bằng `os.walk`.

**Tập khóa (chỉ văn bản)**

`holdout_text_only.csv`, `focused_text_only.csv`, `lazada_text_only.csv` chỉ được phép có hai cột `review_id` và `text`, và không
được trùng review với Train hay Validation.

In [ ]:
def find_file(name, root='/kaggle/input'):
    hits = [os.path.join(d, name) for d, _, fs in os.walk(root) if name in fs]
    if not hits:
        listing = [os.path.join(d, f) for d, _, fs in os.walk(root) for f in fs]
        raise FileNotFoundError(f'Không tìm thấy {name} trong {root}. Các file hiện có:\n' + ('\n'.join(listing) or '(trống)'))
    return hits[0]

labeled_locked = [os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs if f in LOCKED.values()]
assert not labeled_locked, f'Phát hiện file CÓ NHÃN của tập khóa trên Kaggle: {labeled_locked}'
train_df = pd.read_csv(find_file('shopee_train.csv'), encoding='utf-8-sig')
val_df = pd.read_csv(find_file('shopee_validation.csv'), encoding='utf-8-sig')
LOCKED_DF = {name: pd.read_csv(find_file(TEXT_ONLY.format(name)), encoding='utf-8-sig') for name in LOCKED}
seen = set(train_df['review_id']) | set(val_df['review_id'])
for name, df in LOCKED_DF.items():
    assert list(df.columns) == ['review_id', 'text'], f'{name}: file trên Kaggle phải chỉ có review_id và text'
    assert not set(df['review_id']) & seen, f'{name}: trùng review với Train/Validation'
print('Train', len(train_df), '| Validation', len(val_df), '|', {k: len(v) for k, v in LOCKED_DF.items()})
assert (len(train_df), len(val_df)) == (1200, 350)
assert {k: len(v) for k, v in LOCKED_DF.items()} == {'holdout': 1000, 'focused': 150, 'lazada': 500}

## Kiểm tra tokenizer ViSoBERT

**Hai phép kiểm tra**

`self_check` so cách tách với `spm.encode` gốc; `embedding_alignment` che từng token để đầu MLM của ViSoBERT đoán lại. Ở Giai đoạn 6
tỉ lệ đoán đúng là 42,2% khi id trỏ đúng embedding; notebook dừng nếu tỉ lệ dưới 0,3.

In [ ]:
from src.classification.strategy_visobert.tokenizer import VisobertTokenizer, self_check, embedding_alignment
VIS_NAME = FZ['tier1']['visobert']['config']['model_name']
tok = VisobertTokenizer(VIS_NAME)
self_check(tok)
align = embedding_alignment(tok, VIS_NAME, train_df['text'].astype(str).sample(60, random_state=42))
print(f"MLM đoán lại token bị che: {align['mlm_top1']:.1%} trên {align['n_tokens']} token")
assert align['mlm_top1'] > 0.3, 'id token không khớp embedding của ViSoBERT'
del tok; gc.collect(); torch.cuda.empty_cache()

## Train lại từ cấu hình đóng băng

**Năm nhóm mô hình × ba seed**

Tầng 1 (ViSoBERT, Bi-LSTM), tầng 2 (ViSoBERT, Bi-LSTM) và baseline phẳng (ViSoBERT). Mỗi lần train dự đoán luôn Validation và ba tập
khóa; dự đoán của tập khóa chỉ được giữ trong bộ nhớ cho tới khi qua cổng Bước 0.

**Kiểm tra cấu hình**

Runner nhận cấu hình đầy đủ từ file frozen; trước mỗi nhóm có assert cấu hình hiệu lực trùng file frozen.

**Dữ liệu train từng tầng**

Tầng 1: Train 1.200 (gồm ambiguous). Tầng 2: Train actionable, bỏ ambiguous. Baseline: Train 1.200 với nhãn 4 lớp. Từ vựng
Bi-LSTM dựng từ text Train + Validation như ở GĐ6–GĐ7.

In [ ]:
from src.classification.rq1 import VisobertRunner, BilstmRunner, run_multi
from src.classification.tier1_actionable.mapping import to_tier1, TIER1_LABELS
from src.classification.tier2_bfu.mapping import to_tier2, TIER2_LABELS
from src.classification.flat_baseline.mapping import to_flat, FLAT_LABELS
from src.classification.strategy_bilstm_fasttext.embeddings import build_vocab, download, load_vectors

BCFG = FZ['tier1']['bilstm']['config']
vec = download(BCFG['fasttext_url'], '/tmp/cc.vi.300.vec.gz')
vocab = build_vocab(pd.concat([train_df['text'], val_df['text']]).astype(str))
emb, coverage = load_vectors(vec, vocab, dim=int(BCFG['embedding_dim']))
print(f'Từ vựng {len(vocab):,} | có vector fastText: {coverage:.1%}')

SETS = {'validation': val_df, **LOCKED_DF}
SPECS = [('tier1', 'visobert', to_tier1(train_df), TIER1_LABELS), ('tier1', 'bilstm', to_tier1(train_df), TIER1_LABELS),
         ('tier2', 'visobert', to_tier2(train_df, drop_ambiguous=True), TIER2_LABELS),
         ('tier2', 'bilstm', to_tier2(train_df, drop_ambiguous=True), TIER2_LABELS),
         ('flat', 'visobert', to_flat(train_df), FLAT_LABELS)]
RUNS = {}
for tier, model, tr, names in SPECS:
    cfg = FZ[tier][model]['config']
    runner = VisobertRunner(cfg) if model == 'visobert' else BilstmRunner(cfg, vocab, emb)
    assert runner.config(None) == cfg, f'{tier}/{model}: cấu hình hiệu lực khác file frozen'
    for s in SEEDS:
        RUNS[(tier, model, s)] = run_multi(runner, tr, SETS, len(names), names, s)
        r = RUNS[(tier, model, s)]
        print(f"{tier}/{model}/seed {s}: train {r['train_seconds']:.0f}s | dự đoán {r['infer_seconds_per_1000']:.2f}s/1000 | "
              f"GPU {r['peak_gpu_gb']} GB | n_train {r['n_train']}")

## Cổng Bước 0 — so Validation với GĐ6, GĐ7, GĐ9

**Chỉ số so sánh**

Tầng 1: Macro-F1 hai lớp ở ngưỡng đóng băng. Tầng 2: Macro-F1 ba lớp trên gold-actionable. Baseline: Macro-F1 bốn lớp. Giá trị tham
chiếu được đọc từ các bảng kết quả trong repo.

**Quy tắc**

Đạt khi |chênh trung bình 3 seed| ≤ max(2 điểm, độ lệch chuẩn giữa seed của kết quả cũ), và recall ACTIONABLE của seed 42 ở ngưỡng
đóng băng ≥ 0,80 cho cả hai mô hình tầng 1.

In [ ]:
from src.classification import evaluate as E
from src.classification.tier1_actionable.threshold import metrics_at
from src.classification.final_eval import gate_table, parse_pm
RES = os.path.join(REPO_DIR, '05_experiments', 'rq1_classification')
ref1 = pd.read_csv(os.path.join(RES, 'tier1_validation_results.csv')).set_index('strategy')
ref2 = pd.read_csv(os.path.join(RES, 'tier2_validation_results.csv')).set_index('strategy')
reff = pd.read_csv(os.path.join(RES, 'flat_validation_results.csv')).set_index('strategy')
NAME = {'visobert': 'ViSoBERT', 'bilstm': 'Bi-LSTM + fastText'}
y1 = to_tier1(val_df)['y'].to_numpy()
v2 = to_tier2(val_df, drop_ambiguous=False)
mask2 = val_df['review_id'].isin(v2['review_id']).to_numpy()
y2, y4 = v2['y'].to_numpy(), to_flat(val_df)['y'].to_numpy()
MIN_RECALL = FZ['tier1']['threshold_rule']['min_recall']

rows, rec_rows = [], []
for model in ('visobert', 'bilstm'):
    thr = FZ['tier1'][model]['threshold']
    m = [metrics_at(y1, RUNS[('tier1', model, s)]['probs']['validation'][:, 1], thr) for s in SEEDS]
    mf = np.array([x['macro_f1'] for x in m]) * 100
    ref, sd = parse_pm(ref1.loc[NAME[model], 'macro_f1'])
    rows.append({'model': f'Tầng 1 – {NAME[model]} (ngưỡng {thr})', 'ref': ref, 'ref_std': sd, 'new_mean': mf.mean(), 'new_std': mf.std(ddof=1)})
    rec_rows.append({'model': f'Tầng 1 – {NAME[model]}', 'recall ACTIONABLE seed 42': round(m[0]['recall_actionable'], 3),
                     'đạt sàn 0,80': m[0]['recall_actionable'] >= MIN_RECALL})
for model in ('visobert', 'bilstm'):
    mf = np.array([E.metrics(y2, RUNS[('tier2', model, s)]['probs']['validation'][mask2].argmax(1), TIER2_LABELS)['macro_f1']
                   for s in SEEDS]) * 100
    ref, sd = parse_pm(ref2.loc[NAME[model], 'macro_f1'])
    rows.append({'model': f'Tầng 2 – {NAME[model]}', 'ref': ref, 'ref_std': sd, 'new_mean': mf.mean(), 'new_std': mf.std(ddof=1)})
mf = np.array([E.metrics(y4, RUNS[('flat', 'visobert', s)]['probs']['validation'].argmax(1), FLAT_LABELS)['macro_f1']
               for s in SEEDS]) * 100
ref, sd = parse_pm(reff.loc['Baseline phẳng ViSoBERT', 'macro_f1'])
rows.append({'model': 'Baseline phẳng – ViSoBERT', 'ref': ref, 'ref_std': sd, 'new_mean': mf.mean(), 'new_std': mf.std(ddof=1)})

GATE, REC = gate_table(rows), pd.DataFrame(rec_rows)
GATE_PASS = bool(GATE['đạt'].all() and REC['đạt sàn 0,80'].all())
print('Macro-F1 Validation (%): tham chiếu GĐ6/7/9 và mô hình train lại (3 seed)')
print(GATE.round(2).to_string(index=False))
print(); print(REC.to_string(index=False))
print('\nCỔNG BƯỚC 0:', 'ĐẠT — được ghi dự đoán tập khóa' if GATE_PASS else 'KHÔNG ĐẠT — DỪNG, báo lại trước khi mở Hold-out')

## Lưu kết quả

**Luôn lưu**

Dự đoán Validation, bảng cổng Bước 0 và chi phí.

**Chỉ lưu khi qua cổng**

Dự đoán của Hold-out, focused set và Lazada (xác suất mọi seed). Nếu cổng không đạt, cell dừng bằng lỗi và các dự đoán này không
được ghi ra.

In [ ]:
from src.utils.kaggle_paths import working_dir
from src.classification.final_eval import predictions_long
OUT = working_dir('rq1_gd10')
predictions_long(val_df['review_id'], RUNS, 'validation').to_csv(os.path.join(OUT, 'gd10_pred_validation.csv'), index=False)
GATE.to_csv(os.path.join(OUT, 'gd10_gate_validation.csv'), index=False)
REC.to_csv(os.path.join(OUT, 'gd10_gate_recall.csv'), index=False)
cost = []
for tier, model, tr, names in SPECS:
    rs = [RUNS[(tier, model, s)] for s in SEEDS]
    cost.append({'tier': tier, 'model': model, 'n_train': rs[0]['n_train'],
                 'train_seconds_mean': np.mean([r['train_seconds'] for r in rs]),
                 'infer_seconds_per_1000_mean': np.mean([r['infer_seconds_per_1000'] for r in rs]),
                 'peak_gpu_gb_max': max((r['peak_gpu_gb'] or 0) for r in rs),
                 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'})
pd.DataFrame(cost).to_csv(os.path.join(OUT, 'gd10_costs_neural.csv'), index=False)
json.dump({'run_date': datetime.date.today().isoformat(), 'commit': COMMIT, 'gate_pass': GATE_PASS,
           'transformers': transformers.__version__, 'sentencepiece': sentencepiece.__version__},
          open(os.path.join(OUT, 'gd10_run_info.json'), 'w'), indent=1)
if not GATE_PASS:
    raise RuntimeError('Cổng Bước 0 không đạt: KHÔNG ghi dự đoán tập khóa. Dừng và báo lại trước khi mở Hold-out.')
for name, df in LOCKED_DF.items():
    predictions_long(df['review_id'], RUNS, name).to_csv(os.path.join(OUT, f'gd10_pred_{name}.csv'), index=False)
print('Đã lưu:', sorted(os.listdir(OUT)))

## Ghi chú

**Bước tiếp theo**

Tải thư mục `rq1_gd10/` về `05_experiments/rq1_classification/`, rồi chạy notebook 10b (LLM) và cuối cùng là notebook 10 (chấm điểm).
Notebook này không chứa chỉ số nào trên Hold-out, focused set hay Lazada.